In [0]:
df_customers = spark.table("dev.silver.silver_customers")

In [0]:
def customers_dataQuality(df_customers):
    """
    Performs comprehensive data quality checks on customer data.
    Returns a dictionary with quality check results and issue counts.
    """
    from pyspark.sql.functions import col, count, when, countDistinct, lit
    
    quality_results = {}
    total_records = df_customers.count()
    
    print("="*80)
    print("CUSTOMER DATA QUALITY CHECK REPORT")
    print("="*80)
    print(f"\nTotal Records: {total_records:,}\n")
    
    # ========================================================================
    # CHECK 1: Customer ID Uniqueness
    # Description: Ensures each customer has a unique identifier
    # ========================================================================
    print("\n[CHECK 1] Customer ID Uniqueness")
    print("-" * 80)
    print("Description: Verifies that customer_id is unique across all records.")
    print("             Duplicate IDs indicate data integrity issues.\n")
    
    distinct_ids = df_customers.select("customer_id").distinct().count()
    duplicate_ids = total_records - distinct_ids
    
    quality_results['unique_customer_ids'] = {
        'total_records': total_records,
        'distinct_ids': distinct_ids,
        'duplicates': duplicate_ids,
        'passed': duplicate_ids == 0
    }
    
    if duplicate_ids == 0:
        print(f"✓ PASSED: All {total_records:,} customer IDs are unique")
    else:
        print(f"✗ FAILED: Found {duplicate_ids:,} duplicate customer IDs")
        print(f"  Total Records: {total_records:,}")
        print(f"  Distinct IDs: {distinct_ids:,}")
        
        # Show duplicate IDs
        print("\n  Sample Duplicate IDs:")
        df_customers.groupBy("customer_id").count() \
            .filter(col("count") > 1) \
            .orderBy(col("count").desc()) \
            .limit(5) \
            .show(truncate=False)
    
    
    # ========================================================================
    # CHECK 2: Required Attributes - Completeness
    # Description: Validates that critical customer attributes are not null/empty
    # ========================================================================
    print("\n[CHECK 2] Required Attributes - Completeness")
    print("-" * 80)
    print("Description: Checks that essential customer fields contain valid data.")
    print("             Missing values can impact downstream analytics and operations.\n")
    
    required_fields = [
        'customer_id', 'first_name', 'last_name', 'email', 'phone',
        'address', 'city', 'state', 'postal_code', 'customer_segment',
        'customer_status', 'date_of_birth', 'registration_date'
    ]
    
    completeness_results = {}
    all_passed = True
    
    # Get column types
    column_types = dict(df_customers.dtypes)
    
    for field in required_fields:
        # Handle numeric and date columns differently - only check for null
        if column_types.get(field) in ['bigint', 'int', 'long', 'double', 'float', 'date', 'timestamp']:
            null_count = df_customers.filter(
                col(field).isNull()
            ).count()
        else:
            # String columns check for null or empty
            null_count = df_customers.filter(
                col(field).isNull() | (col(field) == "")
            ).count()
        
        completeness_pct = ((total_records - null_count) / total_records * 100) if total_records > 0 else 0
        
        completeness_results[field] = {
            'null_count': null_count,
            'completeness_pct': completeness_pct,
            'passed': null_count == 0
        }
        
        if null_count > 0:
            all_passed = False
    
    quality_results['required_attributes'] = {
        'fields_checked': len(required_fields),
        'all_passed': all_passed,
        'details': completeness_results
    }
    
    if all_passed:
        print(f"✓ PASSED: All {len(required_fields)} required attributes are complete (100%)")
    else:
        print(f"✗ FAILED: Some required attributes have missing values\n")
        print(f"  {'Field':<25} {'Missing':<12} {'Completeness':<15} {'Status'}")
        print("  " + "-" * 70)
        
        for field, result in completeness_results.items():
            status = "✓ PASS" if result['passed'] else "✗ FAIL"
            print(f"  {field:<25} {result['null_count']:<12,} {result['completeness_pct']:>6.2f}%         {status}")
    
    
    # ========================================================================
    # CHECK 3: Email Format Validation
    # Description: Ensures email addresses follow standard format
    # ========================================================================
    print("\n[CHECK 3] Email Format Validation")
    print("-" * 80)
    print("Description: Validates that email addresses contain '@' and domain.")
    print("             Invalid formats may cause communication failures.\n")
    
    email_pattern = "^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}$"
    invalid_emails = df_customers.filter(
        ~col("email").rlike(email_pattern)
    ).count()
    
    quality_results['email_format'] = {
        'invalid_count': invalid_emails,
        'validity_pct': ((total_records - invalid_emails) / total_records * 100) if total_records > 0 else 0,
        'passed': invalid_emails == 0
    }
    
    if invalid_emails == 0:
        print(f"✓ PASSED: All {total_records:,} email addresses are valid")
    else:
        print(f"✗ FAILED: Found {invalid_emails:,} invalid email formats")
        print(f"  Valid Emails: {total_records - invalid_emails:,}")
        print(f"  Validity Rate: {quality_results['email_format']['validity_pct']:.2f}%")
    
    
    # ========================================================================
    # CHECK 4: Phone Number Validation
    # Description: Validates phone numbers are numeric and not null
    # ========================================================================
    print("\n[CHECK 4] Phone Number Validation")
    print("-" * 80)
    print("Description: Ensures phone numbers are valid and properly formatted.")
    print("             Critical for customer contact and verification.\n")
    
    invalid_phones = df_customers.filter(
        col("phone").isNull() | (col("phone") <= 0)
    ).count()
    
    quality_results['phone_validation'] = {
        'invalid_count': invalid_phones,
        'validity_pct': ((total_records - invalid_phones) / total_records * 100) if total_records > 0 else 0,
        'passed': invalid_phones == 0
    }
    
    if invalid_phones == 0:
        print(f"✓ PASSED: All {total_records:,} phone numbers are valid")
    else:
        print(f"✗ FAILED: Found {invalid_phones:,} invalid phone numbers")
        print(f"  Valid Phones: {total_records - invalid_phones:,}")
        print(f"  Validity Rate: {quality_results['phone_validation']['validity_pct']:.2f}%")
    
    
    # ========================================================================
    # CHECK 5: Customer Status Validation
    # Description: Validates customer_status contains only expected values
    # ========================================================================
    print("\n[CHECK 5] Customer Status Validation")
    print("-" * 80)
    print("Description: Checks that customer_status contains only valid categories.")
    print("             Invalid statuses may indicate data entry errors.\n")
    
    valid_statuses = ['Active', 'Inactive', 'Suspended', 'Closed']
    invalid_statuses = df_customers.filter(
        ~col("customer_status").isin(valid_statuses)
    ).count()
    
    quality_results['status_validation'] = {
        'invalid_count': invalid_statuses,
        'valid_values': valid_statuses,
        'passed': invalid_statuses == 0
    }
    
    if invalid_statuses == 0:
        print(f"✓ PASSED: All customer statuses are valid")
        print(f"  Valid values: {', '.join(valid_statuses)}")
    else:
        print(f"✗ FAILED: Found {invalid_statuses:,} invalid customer statuses")
        print(f"  Expected values: {', '.join(valid_statuses)}")
        
        print("\n  Invalid Status Distribution:")
        df_customers.filter(
            ~col("customer_status").isin(valid_statuses)
        ).groupBy("customer_status").count() \
            .orderBy(col("count").desc()) \
            .show(10, truncate=False)
    
    
    # ========================================================================
    # CHECK 6: Date Validity
    # Description: Validates that dates are not in the future or too old
    # ========================================================================
    print("\n[CHECK 6] Date Validity")
    print("-" * 80)
    print("Description: Ensures date_of_birth and registration_date are logical.")
    print("             Future dates or very old dates indicate data quality issues.\n")
    
    from pyspark.sql.functions import current_date
    
    invalid_dob = df_customers.filter(
        (col("date_of_birth") > current_date()) |
        (col("date_of_birth") < lit("1900-01-01"))
    ).count()
    
    invalid_reg_date = df_customers.filter(
        col("registration_date") > current_date()
    ).count()
    
    quality_results['date_validity'] = {
        'invalid_dob': invalid_dob,
        'invalid_registration': invalid_reg_date,
        'passed': (invalid_dob == 0 and invalid_reg_date == 0)
    }
    
    if invalid_dob == 0 and invalid_reg_date == 0:
        print(f"✓ PASSED: All dates are valid")
    else:
        if invalid_dob > 0:
            print(f"✗ FAILED: Found {invalid_dob:,} invalid date_of_birth values")
        if invalid_reg_date > 0:
            print(f"✗ FAILED: Found {invalid_reg_date:,} invalid registration_date values")
    
    
    # ========================================================================
    # CHECK 7: Postal Code Format
    # Description: Validates postal codes are numeric and reasonable length
    # ========================================================================
    print("\n[CHECK 7] Postal Code Format")
    print("-" * 80)
    print("Description: Checks that postal codes are valid integers.")
    print("             Important for location-based analytics and shipping.\n")
    
    invalid_postal = df_customers.filter(
        col("postal_code").isNull() | 
        (col("postal_code") <= 0) |
        (col("postal_code") > 999999)
    ).count()
    
    quality_results['postal_code_format'] = {
        'invalid_count': invalid_postal,
        'validity_pct': ((total_records - invalid_postal) / total_records * 100) if total_records > 0 else 0,
        'passed': invalid_postal == 0
    }
    
    if invalid_postal == 0:
        print(f"✓ PASSED: All {total_records:,} postal codes are valid")
    else:
        print(f"✗ FAILED: Found {invalid_postal:,} invalid postal codes")
        print(f"  Valid Postal Codes: {total_records - invalid_postal:,}")
        print(f"  Validity Rate: {quality_results['postal_code_format']['validity_pct']:.2f}%")
    
    
    # ========================================================================
    # OVERALL SUMMARY
    # ========================================================================
    print("\n" + "="*80)
    print("OVERALL DATA QUALITY SUMMARY")
    print("="*80 + "\n")
    
    checks_passed = sum([
        quality_results['unique_customer_ids']['passed'],
        quality_results['required_attributes']['all_passed'],
        quality_results['email_format']['passed'],
        quality_results['phone_validation']['passed'],
        quality_results['status_validation']['passed'],
        quality_results['date_validity']['passed'],
        quality_results['postal_code_format']['passed']
    ])
    
    total_checks = 7
    quality_score = (checks_passed / total_checks * 100)
    
    print(f"Total Quality Checks: {total_checks}")
    print(f"Checks Passed: {checks_passed}")
    print(f"Checks Failed: {total_checks - checks_passed}")
    print(f"\nOverall Data Quality Score: {quality_score:.1f}%\n")
    
    if quality_score == 100:
        print("🎉 EXCELLENT: All data quality checks passed!")
    elif quality_score >= 80:
        print("✓ GOOD: Most quality checks passed, minor issues detected.")
    elif quality_score >= 60:
        print("⚠ WARNING: Multiple quality issues detected, attention needed.")
    else:
        print("✗ CRITICAL: Significant data quality issues require immediate action.")
    
    print("\n" + "="*80 + "\n")
    
    return quality_results

In [0]:
# Execute data quality checks on customer data
quality_report = customers_dataQuality(df_customers)

In [0]:
def account_dataQuality(df_accounts, df_customers, df_branches):
    """
    Performs comprehensive data quality checks on account data.
    Validates relationships with customers and branches.
    Returns a dictionary with quality check results and issue counts.
    """
    from pyspark.sql.functions import col, count, when, countDistinct, lit
    
    quality_results = {}
    total_records = df_accounts.count()
    
    print("="*80)
    print("ACCOUNT DATA QUALITY CHECK REPORT")
    print("="*80)
    print(f"\nTotal Records: {total_records:,}\n")
    
    # ========================================================================
    # CHECK 1: Account ID Uniqueness
    # Description: Ensures each account has a unique identifier
    # ========================================================================
    print("\n[CHECK 1] Account ID Uniqueness")
    print("-" * 80)
    print("Description: Verifies that account_id is unique across all records.")
    print("             Duplicate IDs indicate data integrity issues.\n")
    
    distinct_ids = df_accounts.select("account_id").distinct().count()
    duplicate_ids = total_records - distinct_ids
    
    quality_results['unique_account_ids'] = {
        'total_records': total_records,
        'distinct_ids': distinct_ids,
        'duplicates': duplicate_ids,
        'passed': duplicate_ids == 0
    }
    
    if duplicate_ids == 0:
        print(f"✓ PASSED: All {total_records:,} account IDs are unique")
    else:
        print(f"✗ FAILED: Found {duplicate_ids:,} duplicate account IDs")
        print(f"  Total Records: {total_records:,}")
        print(f"  Distinct IDs: {distinct_ids:,}")
        
        # Show duplicate IDs
        print("\n  Sample Duplicate IDs:")
        df_accounts.groupBy("account_id").count() \
            .filter(col("count") > 1) \
            .orderBy(col("count").desc()) \
            .limit(5) \
            .show(truncate=False)
    
    
    # ========================================================================
    # CHECK 2: Customer Reference Integrity
    # Description: Every valid account should reference an existing customer
    # ========================================================================
    print("\n[CHECK 2] Customer Reference Integrity")
    print("-" * 80)
    print("Description: Validates that every account references a valid customer.")
    print("             Orphaned accounts cannot be attributed to any customer.\n")
    
    # Get list of valid customer IDs
    valid_customer_ids = df_customers.select("customer_id").distinct()
    
    # Find accounts with invalid customer references
    accounts_with_invalid_customers = df_accounts.join(
        valid_customer_ids,
        df_accounts.customer_id == valid_customer_ids.customer_id,
        "left_anti"
    ).count()
    
    quality_results['customer_reference'] = {
        'invalid_count': accounts_with_invalid_customers,
        'validity_pct': ((total_records - accounts_with_invalid_customers) / total_records * 100) if total_records > 0 else 0,
        'passed': accounts_with_invalid_customers == 0
    }
    
    if accounts_with_invalid_customers == 0:
        print(f"✓ PASSED: All {total_records:,} accounts reference valid customers")
    else:
        print(f"✗ FAILED: Found {accounts_with_invalid_customers:,} accounts with invalid customer references")
        print(f"  Valid References: {total_records - accounts_with_invalid_customers:,}")
        print(f"  Validity Rate: {quality_results['customer_reference']['validity_pct']:.2f}%")
        
        # Show sample orphaned accounts
        print("\n  Sample Orphaned Accounts (invalid customer_id):")
        df_accounts.join(
            valid_customer_ids,
            df_accounts.customer_id == valid_customer_ids.customer_id,
            "left_anti"
        ).select("account_id", "customer_id", "account_type", "account_status") \
            .limit(5) \
            .show(truncate=False)
    
    
    # ========================================================================
    # CHECK 3: Branch Reference Integrity
    # Description: Every valid account should reference an existing branch
    # ========================================================================
    print("\n[CHECK 3] Branch Reference Integrity")
    print("-" * 80)
    print("Description: Validates that every account references a valid branch.")
    print("             Accounts without valid branches cannot be located.\n")
    
    # Get list of valid branch IDs
    valid_branch_ids = df_branches.select("branch_id").distinct()
    
    # Find accounts with invalid branch references
    accounts_with_invalid_branches = df_accounts.join(
        valid_branch_ids,
        df_accounts.branch_id == valid_branch_ids.branch_id,
        "left_anti"
    ).count()
    
    quality_results['branch_reference'] = {
        'invalid_count': accounts_with_invalid_branches,
        'validity_pct': ((total_records - accounts_with_invalid_branches) / total_records * 100) if total_records > 0 else 0,
        'passed': accounts_with_invalid_branches == 0
    }
    
    if accounts_with_invalid_branches == 0:
        print(f"✓ PASSED: All {total_records:,} accounts reference valid branches")
    else:
        print(f"✗ FAILED: Found {accounts_with_invalid_branches:,} accounts with invalid branch references")
        print(f"  Valid References: {total_records - accounts_with_invalid_branches:,}")
        print(f"  Validity Rate: {quality_results['branch_reference']['validity_pct']:.2f}%")
        
        # Show sample accounts with invalid branches
        print("\n  Sample Accounts with Invalid branch_id:")
        df_accounts.join(
            valid_branch_ids,
            df_accounts.branch_id == valid_branch_ids.branch_id,
            "left_anti"
        ).select("account_id", "branch_id", "customer_id", "account_type") \
            .limit(5) \
            .show(truncate=False)
    
    
    # ========================================================================
    # CHECK 4: Required Attributes - Completeness
    # Description: Validates that critical account attributes are not null/empty
    # ========================================================================
    print("\n[CHECK 4] Required Attributes - Completeness")
    print("-" * 80)
    print("Description: Checks that essential account fields contain valid data.")
    print("             Missing values can impact account operations and reporting.\n")
    
    required_fields = [
        'account_id', 'customer_id', 'branch_id', 'account_type',
        'account_status', 'opening_date'
    ]
    
    completeness_results = {}
    all_passed = True
    
    # Get column types
    column_types = dict(df_accounts.dtypes)
    
    for field in required_fields:
        # Handle numeric and date columns differently - only check for null
        if column_types.get(field) in ['bigint', 'int', 'long', 'double', 'float', 'decimal', 'date', 'timestamp']:
            null_count = df_accounts.filter(
                col(field).isNull()
            ).count()
        else:
            # String columns check for null or empty
            null_count = df_accounts.filter(
                col(field).isNull() | (col(field) == "")
            ).count()
        
        completeness_pct = ((total_records - null_count) / total_records * 100) if total_records > 0 else 0
        
        completeness_results[field] = {
            'null_count': null_count,
            'completeness_pct': completeness_pct,
            'passed': null_count == 0
        }
        
        if null_count > 0:
            all_passed = False
    
    quality_results['required_attributes'] = {
        'fields_checked': len(required_fields),
        'all_passed': all_passed,
        'details': completeness_results
    }
    
    if all_passed:
        print(f"✓ PASSED: All {len(required_fields)} required attributes are complete (100%)")
    else:
        print(f"✗ FAILED: Some required attributes have missing values\n")
        print(f"  {'Field':<25} {'Missing':<12} {'Completeness':<15} {'Status'}")
        print("  " + "-" * 70)
        
        for field, result in completeness_results.items():
            status = "✓ PASS" if result['passed'] else "✗ FAIL"
            print(f"  {field:<25} {result['null_count']:<12,} {result['completeness_pct']:>6.2f}%         {status}")
    
    
    # ========================================================================
    # CHECK 5: Account Type Validation
    # Description: Validates account_type contains only expected values
    # ========================================================================
    print("\n[CHECK 5] Account Type Validation")
    print("-" * 80)
    print("Description: Checks that account_type contains only valid categories.")
    print("             Invalid types may indicate data entry errors.\n")
    
    valid_types = ['Savings', 'Checking', 'Credit', 'Loan', 'Investment']
    invalid_types = df_accounts.filter(
        ~col("account_type").isin(valid_types)
    ).count()
    
    quality_results['type_validation'] = {
        'invalid_count': invalid_types,
        'valid_values': valid_types,
        'passed': invalid_types == 0
    }
    
    if invalid_types == 0:
        print(f"✓ PASSED: All account types are valid")
        print(f"  Valid values: {', '.join(valid_types)}")
    else:
        print(f"✗ FAILED: Found {invalid_types:,} invalid account types")
        print(f"  Expected values: {', '.join(valid_types)}")
        
        print("\n  Invalid Type Distribution:")
        df_accounts.filter(
            ~col("account_type").isin(valid_types)
        ).groupBy("account_type").count() \
            .orderBy(col("count").desc()) \
            .show(10, truncate=False)
    
    
    # ========================================================================
    # CHECK 6: Account Status Validation
    # Description: Validates account_status contains only expected values
    # ========================================================================
    print("\n[CHECK 6] Account Status Validation")
    print("-" * 80)
    print("Description: Checks that account_status contains only valid categories.")
    print("             Invalid statuses may impact account operations.\n")
    
    valid_statuses = ['Active', 'Inactive', 'Closed', 'Suspended', 'Frozen']
    invalid_statuses = df_accounts.filter(
        ~col("account_status").isin(valid_statuses)
    ).count()
    
    quality_results['status_validation'] = {
        'invalid_count': invalid_statuses,
        'valid_values': valid_statuses,
        'passed': invalid_statuses == 0
    }
    
    if invalid_statuses == 0:
        print(f"✓ PASSED: All account statuses are valid")
        print(f"  Valid values: {', '.join(valid_statuses)}")
    else:
        print(f"✗ FAILED: Found {invalid_statuses:,} invalid account statuses")
        print(f"  Expected values: {', '.join(valid_statuses)}")
        
        print("\n  Invalid Status Distribution:")
        df_accounts.filter(
            ~col("account_status").isin(valid_statuses)
        ).groupBy("account_status").count() \
            .orderBy(col("count").desc()) \
            .show(10, truncate=False)
    
    
    # ========================================================================
    # CHECK 7: Date Validity
    # Description: Validates that opening_date is logical
    # ========================================================================
    print("\n[CHECK 7] Date Validity")
    print("-" * 80)
    print("Description: Ensures opening_date is not in the future or too old.")
    print("             Invalid dates indicate data quality issues.\n")
    
    from pyspark.sql.functions import current_date
    
    invalid_opening_date = df_accounts.filter(
        (col("opening_date") > current_date()) |
        (col("opening_date") < lit("1900-01-01"))
    ).count()
    
    quality_results['date_validity'] = {
        'invalid_opening_date': invalid_opening_date,
        'passed': invalid_opening_date == 0
    }
    
    if invalid_opening_date == 0:
        print(f"✓ PASSED: All opening_date values are valid")
    else:
        print(f"✗ FAILED: Found {invalid_opening_date:,} invalid opening_date values")
    
    
    # ========================================================================
    # OVERALL SUMMARY
    # ========================================================================
    print("\n" + "="*80)
    print("OVERALL DATA QUALITY SUMMARY")
    print("="*80 + "\n")
    
    checks_passed = sum([
        quality_results['unique_account_ids']['passed'],
        quality_results['customer_reference']['passed'],
        quality_results['branch_reference']['passed'],
        quality_results['required_attributes']['all_passed'],
        quality_results['type_validation']['passed'],
        quality_results['status_validation']['passed'],
        quality_results['date_validity']['passed']
    ])
    
    total_checks = 7
    quality_score = (checks_passed / total_checks * 100)
    
    print(f"Total Quality Checks: {total_checks}")
    print(f"Checks Passed: {checks_passed}")
    print(f"Checks Failed: {total_checks - checks_passed}")
    print(f"\nOverall Data Quality Score: {quality_score:.1f}%\n")
    
    if quality_score == 100:
        print("🎉 EXCELLENT: All data quality checks passed!")
    elif quality_score >= 80:
        print("✓ GOOD: Most quality checks passed, minor issues detected.")
    elif quality_score >= 60:
        print("⚠ WARNING: Multiple quality issues detected, attention needed.")
    else:
        print("✗ CRITICAL: Significant data quality issues require immediate action.")
    
    print("\n" + "="*80 + "\n")
    
    return quality_results


In [0]:
# Load accounts, customers, and branches for account quality checks
df_accounts = spark.table("dev.silver.silver_accounts")
df_branches = spark.table("dev.silver.silver_branches")

In [0]:
# Execute data quality checks on account data
account_quality_report = account_dataQuality(df_accounts, df_customers, df_branches)

In [0]:
def transactions_dataQuality(df_transactions, df_accounts):
    """
    Performs comprehensive data quality checks on transaction data.
    Validates relationships with accounts and transaction integrity.
    Returns a dictionary with quality check results and issue counts.
    """
    from pyspark.sql.functions import col, count, when, countDistinct, lit
    
    quality_results = {}
    total_records = df_transactions.count()
    
    print("="*80)
    print("TRANSACTION DATA QUALITY CHECK REPORT")
    print("="*80)
    print(f"\nTotal Records: {total_records:,}\n")
    
    # ========================================================================
    # CHECK 1: Transaction ID Uniqueness
    # Description: Ensures each transaction has a unique identifier
    # ========================================================================
    print("\n[CHECK 1] Transaction ID Uniqueness")
    print("-" * 80)
    print("Description: Verifies that transaction_id is unique across all records.")
    print("             Duplicate IDs indicate data integrity issues.\n")
    
    distinct_ids = df_transactions.select("transaction_id").distinct().count()
    duplicate_ids = total_records - distinct_ids
    
    quality_results['unique_transaction_ids'] = {
        'total_records': total_records,
        'distinct_ids': distinct_ids,
        'duplicates': duplicate_ids,
        'passed': duplicate_ids == 0
    }
    
    if duplicate_ids == 0:
        print(f"✓ PASSED: All {total_records:,} transaction IDs are unique")
    else:
        print(f"✗ FAILED: Found {duplicate_ids:,} duplicate transaction IDs")
        print(f"  Total Records: {total_records:,}")
        print(f"  Distinct IDs: {distinct_ids:,}")
        
        # Show duplicate IDs
        print("\n  Sample Duplicate IDs:")
        df_transactions.groupBy("transaction_id").count() \
            .filter(col("count") > 1) \
            .orderBy(col("count").desc()) \
            .limit(5) \
            .show(truncate=False)
    
    
    # ========================================================================
    # CHECK 2: Transaction Amount Validation
    # Description: Validates that transaction amounts are valid
    # ========================================================================
    print("\n[CHECK 2] Transaction Amount Validation")
    print("-" * 80)
    print("Description: Checks that transaction amounts are not null or zero.")
    print("             Invalid amounts indicate incomplete transaction data.\n")
    
    # Check for null amounts
    null_amounts = df_transactions.filter(
        col("amount").isNull()
    ).count()
    
    # Check for zero or negative amounts (may be valid for refunds/reversals)
    zero_amounts = df_transactions.filter(
        col("amount") == 0
    ).count()
    
    negative_amounts = df_transactions.filter(
        col("amount") < 0
    ).count()
    
    quality_results['amount_validation'] = {
        'null_count': null_amounts,
        'zero_count': zero_amounts,
        'negative_count': negative_amounts,
        'passed': null_amounts == 0
    }
    
    if null_amounts == 0:
        print(f"✓ PASSED: All {total_records:,} transactions have valid amounts")
        if zero_amounts > 0:
            print(f"  INFO: Found {zero_amounts:,} zero-amount transactions (may be valid)")
        if negative_amounts > 0:
            print(f"  INFO: Found {negative_amounts:,} negative transactions (refunds/reversals)")
    else:
        print(f"✗ FAILED: Found {null_amounts:,} transactions with null amounts")
        print(f"  Valid Amounts: {total_records - null_amounts:,}")
    
    
    # ========================================================================
    # CHECK 3: Account Reference Integrity
    # Description: Every valid transaction should reference an existing account
    # ========================================================================
    print("\n[CHECK 3] Account Reference Integrity")
    print("-" * 80)
    print("Description: Validates that every transaction references a valid account.")
    print("             Orphaned transactions cannot be attributed to any account.\n")
    
    # Get list of valid account IDs
    valid_account_ids = df_accounts.select("account_id").distinct()
    
    # Find transactions with invalid account references
    transactions_with_invalid_accounts = df_transactions.join(
        valid_account_ids,
        df_transactions.account_id == valid_account_ids.account_id,
        "left_anti"
    ).count()
    
    quality_results['account_reference'] = {
        'invalid_count': transactions_with_invalid_accounts,
        'validity_pct': ((total_records - transactions_with_invalid_accounts) / total_records * 100) if total_records > 0 else 0,
        'passed': transactions_with_invalid_accounts == 0
    }
    
    if transactions_with_invalid_accounts == 0:
        print(f"✓ PASSED: All {total_records:,} transactions reference valid accounts")
    else:
        print(f"✗ FAILED: Found {transactions_with_invalid_accounts:,} transactions with invalid account references")
        print(f"  Valid References: {total_records - transactions_with_invalid_accounts:,}")
        print(f"  Validity Rate: {quality_results['account_reference']['validity_pct']:.2f}%")
        
        # Show sample orphaned transactions
        print("\n  Sample Orphaned Transactions (invalid account_id):")
        df_transactions.join(
            valid_account_ids,
            df_transactions.account_id == valid_account_ids.account_id,
            "left_anti"
        ).select("transaction_id", "account_id", "amount", "transaction_type", "transaction_date") \
            .limit(5) \
            .show(truncate=False)
    
    
    # ========================================================================
    # CHECK 4: Required Attributes - Completeness
    # Description: Validates that critical transaction attributes are not null/empty
    # ========================================================================
    print("\n[CHECK 4] Required Attributes - Completeness")
    print("-" * 80)
    print("Description: Checks that essential transaction fields contain valid data.")
    print("             Missing values can impact transaction processing and reporting.\n")
    
    required_fields = [
        'transaction_id', 'account_id', 'transaction_type',
        'amount', 'transaction_date', 'transaction_status'
    ]
    
    completeness_results = {}
    all_passed = True
    
    # Get column types
    column_types = dict(df_transactions.dtypes)
    
    for field in required_fields:
        # Handle numeric and date columns differently - only check for null
        if column_types.get(field) in ['bigint', 'int', 'long', 'double', 'float', 'decimal', 'date', 'timestamp']:
            null_count = df_transactions.filter(
                col(field).isNull()
            ).count()
        else:
            # String columns check for null or empty
            null_count = df_transactions.filter(
                col(field).isNull() | (col(field) == "")
            ).count()
        
        completeness_pct = ((total_records - null_count) / total_records * 100) if total_records > 0 else 0
        
        completeness_results[field] = {
            'null_count': null_count,
            'completeness_pct': completeness_pct,
            'passed': null_count == 0
        }
        
        if null_count > 0:
            all_passed = False
    
    quality_results['required_attributes'] = {
        'fields_checked': len(required_fields),
        'all_passed': all_passed,
        'details': completeness_results
    }
    
    if all_passed:
        print(f"✓ PASSED: All {len(required_fields)} required attributes are complete (100%)")
    else:
        print(f"✗ FAILED: Some required attributes have missing values\n")
        print(f"  {'Field':<25} {'Missing':<12} {'Completeness':<15} {'Status'}")
        print("  " + "-" * 70)
        
        for field, result in completeness_results.items():
            status = "✓ PASS" if result['passed'] else "✗ FAIL"
            print(f"  {field:<25} {result['null_count']:<12,} {result['completeness_pct']:>6.2f}%         {status}")
    
    
    # ========================================================================
    # CHECK 5: Transaction Type Validation
    # Description: Validates transaction_type contains only expected values
    # ========================================================================
    print("\n[CHECK 5] Transaction Type Validation")
    print("-" * 80)
    print("Description: Checks that transaction_type contains only valid categories.")
    print("             Invalid types may indicate data entry errors.\n")
    
    valid_types = ['Deposit', 'Withdrawal', 'Transfer', 'Payment', 'Refund', 'Fee']
    invalid_types = df_transactions.filter(
        ~col("transaction_type").isin(valid_types)
    ).count()
    
    quality_results['type_validation'] = {
        'invalid_count': invalid_types,
        'valid_values': valid_types,
        'passed': invalid_types == 0
    }
    
    if invalid_types == 0:
        print(f"✓ PASSED: All transaction types are valid")
        print(f"  Valid values: {', '.join(valid_types)}")
    else:
        print(f"✗ FAILED: Found {invalid_types:,} invalid transaction types")
        print(f"  Expected values: {', '.join(valid_types)}")
        
        print("\n  Invalid Type Distribution:")
        df_transactions.filter(
            ~col("transaction_type").isin(valid_types)
        ).groupBy("transaction_type").count() \
            .orderBy(col("count").desc()) \
            .show(10, truncate=False)
    
    
    # ========================================================================
    # CHECK 6: Transaction Status Validation
    # Description: Validates transaction_status contains only expected values
    # ========================================================================
    print("\n[CHECK 6] Transaction Status Validation")
    print("-" * 80)
    print("Description: Checks that transaction_status contains only valid categories.")
    print("             Invalid statuses may impact transaction processing.\n")
    
    valid_statuses = ['Completed', 'Pending', 'Failed', 'Cancelled', 'Reversed']
    invalid_statuses = df_transactions.filter(
        ~col("transaction_status").isin(valid_statuses)
    ).count()
    
    quality_results['status_validation'] = {
        'invalid_count': invalid_statuses,
        'valid_values': valid_statuses,
        'passed': invalid_statuses == 0
    }
    
    if invalid_statuses == 0:
        print(f"✓ PASSED: All transaction statuses are valid")
        print(f"  Valid values: {', '.join(valid_statuses)}")
    else:
        print(f"✗ FAILED: Found {invalid_statuses:,} invalid transaction statuses")
        print(f"  Expected values: {', '.join(valid_statuses)}")
        
        print("\n  Invalid Status Distribution:")
        df_transactions.filter(
            ~col("transaction_status").isin(valid_statuses)
        ).groupBy("transaction_status").count() \
            .orderBy(col("count").desc()) \
            .show(10, truncate=False)
    
    
    # ========================================================================
    # CHECK 7: Date Validity
    # Description: Validates that transaction_date is logical
    # ========================================================================
    print("\n[CHECK 7] Date Validity")
    print("-" * 80)
    print("Description: Ensures transaction_date is not in the future or too old.")
    print("             Invalid dates indicate data quality issues.\n")
    
    from pyspark.sql.functions import current_date
    
    invalid_transaction_date = df_transactions.filter(
        (col("transaction_date") > current_date()) |
        (col("transaction_date") < lit("1900-01-01"))
    ).count()
    
    quality_results['date_validity'] = {
        'invalid_transaction_date': invalid_transaction_date,
        'passed': invalid_transaction_date == 0
    }
    
    if invalid_transaction_date == 0:
        print(f"✓ PASSED: All transaction_date values are valid")
    else:
        print(f"✗ FAILED: Found {invalid_transaction_date:,} invalid transaction_date values")
    
    
    # ========================================================================
    # OVERALL SUMMARY
    # ========================================================================
    print("\n" + "="*80)
    print("OVERALL DATA QUALITY SUMMARY")
    print("="*80 + "\n")
    
    checks_passed = sum([
        quality_results['unique_transaction_ids']['passed'],
        quality_results['amount_validation']['passed'],
        quality_results['account_reference']['passed'],
        quality_results['required_attributes']['all_passed'],
        quality_results['type_validation']['passed'],
        quality_results['status_validation']['passed'],
        quality_results['date_validity']['passed']
    ])
    
    total_checks = 7
    quality_score = (checks_passed / total_checks * 100)
    
    print(f"Total Quality Checks: {total_checks}")
    print(f"Checks Passed: {checks_passed}")
    print(f"Checks Failed: {total_checks - checks_passed}")
    print(f"\nOverall Data Quality Score: {quality_score:.1f}%\n")
    
    if quality_score == 100:
        print("🎉 EXCELLENT: All data quality checks passed!")
    elif quality_score >= 80:
        print("✓ GOOD: Most quality checks passed, minor issues detected.")
    elif quality_score >= 60:
        print("⚠ WARNING: Multiple quality issues detected, attention needed.")
    else:
        print("✗ CRITICAL: Significant data quality issues require immediate action.")
    
    print("\n" + "="*80 + "\n")
    
    return quality_results

In [0]:
# Load transactions table
df_transactions = spark.table("dev.silver.silver_transactions")

# Execute data quality checks on transaction data
transaction_quality_report = transactions_dataQuality(df_transactions, df_accounts)